In [1]:
import os

In [2]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/zeeljavia123/DataScienceProject1.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="zeeljavia123"
os. environ["MLFLOW_TRACKING_PASSWORD"]="0541d252d57eb03417c366d02c2b97ad879333ef"

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path
import os

In [5]:
@dataclass
class ModelEvaluationConfig:
  root_dir: Path
  test_data_path: Path
  model_path: Path
  all_params : dict
  metric_file_name: Path
  target_column: str
  mlflow_uri: str

In [6]:
from src.DataScienceProject.constants import *
from src.DataScienceProject.utils.common import read_yaml, create_directories , save_json

In [18]:
class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH,
    schema_filepath = SCHEMA_FILE_PATH):

      self.config = read_yaml(config_filepath)
      self.params = read_yaml(params_filepath)
      self. schema = read_yaml(schema_filepath)

      create_directories([self.config.model_trainer.root_dir])
  
  def get_model_evaluation_config(self) -> ModelEvaluationConfig:
    config = self.config.model_evaluation
    params = self.params.ElasticNet
    schema = self.schema.TARGET_COLUMN
    
    create_directories([config.root_dir])

    model_evaluation_config = ModelEvaluationConfig(
      root_dir = Path(config.root_dir),
      test_data_path = Path(config.test_data_path),
      model_path = Path(config.model_path),
      all_params = params,
      metric_file_name = Path(config.metric_file_name),
      target_column = schema.name,
      mlflow_uri ='https://dagshub.com/zeeljavia123/DataScienceProject1.mlflow'
      
    )

    return model_evaluation_config

In [8]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib


d:\MLOPS\Projects\DataScienceProject1\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelEvaluation:
  def __init__(self, config: ModelEvaluationConfig):
    self.config = config
  
  def eval_metrics(self, actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    return rmse, mae, r2
  
  def log_into_mlflow(self):
    test_data = pd.read_csv(self.config.test_data_path)
    model = joblib.load(self.config.model_path)
    
    test_x = test_data.drop(self.config.target_column, axis=1)
    test_y = test_data[self.config.target_column]
    
    mlflow.set_tracking_uri(self.config.mlflow_uri)
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    
    with mlflow.start_run():

      predicted_qualities = model.predict(test_x)

      (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

      # Saving metrics as local
      scores = {"rmse": rmse, "mae": mae, "r2": r2}
      save_json(path=Path(self.config.metric_file_name), data=scores)

      mlflow. log_params(self.config.all_params)

      mlflow.log_metric("rmse", rmse)
      mlflow.log_metric("r2", r2)
      mlflow. log_metric("mae", mae)

      # Model registry does not work with file store
      if tracking_url_type_store != "file":
        # Register the model
        # There are other ways to use the Model Registry, which depends on the use case,
        # please refer to the doc for more information:
        # https://mlflow.org/docs/latest/model-registry.html#api-workflow
        mlflow.sklearn. log_model(model, "model", registered_model_name="ElasticnetModel")
      else:
        mlflow.sklearn.log_model(model,"model")

In [10]:
import os

print(os.path.exists("artifacts/model_trainer/model.joblib"))

False


In [11]:
from src.DataScienceProject import logger

In [20]:
try:
    config = ConfigurationManager()

    # Get model evaluation configuration
    model_evaluation_config = config.get_model_evaluation_config()

    # Create ModelEvaluation object
    model_evaluation = ModelEvaluation(config=model_evaluation_config)

    # Run MLflow logging
    model_evaluation.log_into_mlflow()

except Exception as e:
    logger.exception(f"Model evaluation failed: {e}")

[2026-05-20 08:37:12,895]: INFO: common :yaml file: config\config.yaml loaded successfully 
[2026-05-20 08:37:12,900]: INFO: common :yaml file: params.yaml loaded successfully 
[2026-05-20 08:37:12,907]: INFO: common :yaml file: schema.yaml loaded successfully 
[2026-05-20 08:37:12,911]: INFO: common :created directory at: artifacts/model_trainer 
[2026-05-20 08:37:12,914]: INFO: common :created directory at: artifacts/model_evaluation 
[2026-05-20 08:37:26,632]: INFO: common :json file saved at: artifacts\model_evaluation\metrics.json 


2026/05/20 08:37:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 08:37:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetModel'.
2026/05/20 08:38:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run brawny-steed-181 at: https://dagshub.com/zeeljavia123/DataScienceProject1.mlflow/#/experiments/0/runs/9b427e97d757474e98dc762ec48da252
🧪 View experiment at: https://dagshub.com/zeeljavia123/DataScienceProject1.mlflow/#/experiments/0


In [13]:
joblib.dump(model, self.config.model_path)

NameError: name 'model' is not defined

In [ ]:
%pwd

'd:\\MLOPS\\Projects'

In [ ]:
import os

In [ ]:
%pwd

'd:\\MLOPS\\Projects'

In [ ]:
os.chdir("D:/MLOPS/Projects/DataScienceProject1")

In [ ]:
%pwd

'D:\\MLOPS\\Projects\\DataScienceProject1'